In [ ]:
import pandas as pd
import os


In [ ]:
import statistics as s
from math import isnan
from itertools import filterfalse
import numpy as np

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)

In [ ]:
#getting csv with the viral concepts that will be chorts
concepts_csv = '/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/viral_cohort_patient_overlap_seasonal_nonseasonal_update1.csv'
viral_concept_df = pd.read_csv(concepts_csv)

In [ ]:
viral_concept_df.head(100)

In [ ]:
#seasonal+vaccination  DF

In [ ]:
seasonal_table = viral_concept_df.loc[:, ['concept_id','standard_concept_name','count','seasonal_vax', 'specie', 'genus'] ]
seasonal_table = seasonal_table.loc[seasonal_table['seasonal_vax'] == 'Y', :]
seasonal_table

In [ ]:
def merge_specie_data(new_column):
    
    if new_column["genus"] == "Influenzavirus":
        return new_column["genus"]
    else:
        return new_column["specie"]
    

    
seasonal_table["updated_specie"] = seasonal_table.apply(merge_specie_data, axis=1)
seasonal_vax_table = seasonal_table.loc[:, ['concept_id','standard_concept_name', 'updated_specie']]
seasonal_vax_table 

In [ ]:
#mapping specie:vaccine list
concepts_csv1 = '/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/ns_viral_cohort_vax_ids.csv'
vaccine_concepts = pd.read_csv(concepts_csv1)
vaccine_concept_df = vaccine_concepts.loc[:, ['updated_specie', 'vaccine_id']]
vaccine_concept_df

In [ ]:
seasonal_cohort_vaccine_df = seasonal_vax_table.merge(vaccine_concept_df,  on='updated_specie', how='left')
seasonal_cohort_vaccine_df

In [ ]:
def get_concept_vax_id_dict(ns_cohort_vax_df): 
    '''
       1. subset a new df with only 2 columns: concept_id and vaccine_id
       2. reset the index to concept_id
       3. groupby concept_id, selecting (vaccine_id) col as a series, and using .agg to apply an aggregation to each 
        group, that being the lambda function to make the grouped series into a single list
    
    ''' 
    
    df1 = seasonal_cohort_vaccine_df.loc[ :, ['concept_id', 'vaccine_id']]
    df2 = df1.set_index('concept_id')
    df3 = df2.groupby('concept_id', sort=False)['vaccine_id'].agg(lambda vax_id: list(vax_id))
    final = df3.to_dict()
    
    
    return final

cohort_vax_dict = get_concept_vax_id_dict(seasonal_cohort_vaccine_df)
cohort_vax_dict

In [ ]:
def get_condition_summary(concept_id):
    """
    Fetches per-patient summary for the specified condition_concept_id,
    applying Cohort Builder UI filters (EHR + genomics, observation window,
    flat-events, standard concepts), and returns a DataFrame with one row per
    patient including:
      - condition_concept_id
      - standard_concept_name, standard_vocabulary
      - first and last diagnosis date for that concept
      - condition_type_concept_name, visit_occurrence_concept_name from first occurrence
      - visits_for_concept: count of unique visit_occurrence_id for the concept
      - visits_all_concepts: count of unique visits across all conditions
      - concept_count_ehr: count of distinct condition concepts in EHR
    """
    dataset = os.environ["WORKSPACE_CDR"]
    sql = f"""
    WITH
      ehr_genomics_patients AS (
        SELECT DISTINCT person_id
        FROM `{dataset}.cb_search_person`
        WHERE has_ehr_data = 1
          AND (
               has_whole_genome_variant      = 1
            OR has_lr_whole_genome_variant   = 1
            OR has_array_data                = 1
          )
      ),

      all_occ AS (
        SELECT
          co.person_id,
          co.condition_concept_id,
          co.visit_occurrence_id,
          co.condition_start_datetime,
          co.condition_end_datetime,
          co.condition_type_concept_id
        FROM `{dataset}.condition_occurrence` co
        JOIN ehr_genomics_patients eg
          ON co.person_id = eg.person_id

        -- only events that made it into the CB search table
        JOIN `{dataset}.cb_search_all_events` ev
          ON ev.person_id = co.person_id
         AND ev.concept_id = co.condition_concept_id
         AND DATE(co.condition_start_datetime) = ev.entry_date

        JOIN `{dataset}.concept` c_std
          ON co.condition_concept_id = c_std.concept_id
        WHERE c_std.standard_concept = 'S'
      ),

      spec_occ AS (
        SELECT *
        FROM all_occ
        WHERE condition_concept_id = {concept_id}
      ),

      detail AS (
        SELECT
          person_id,
          condition_concept_id,
          condition_start_datetime AS first_diag_date,
          condition_end_datetime   AS first_end_date,
          condition_type_concept_id,
          visit_occurrence_id,
          ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY condition_start_datetime) AS rn
        FROM spec_occ
      ),

      first_detail AS (
        SELECT
          d.person_id,
          d.condition_concept_id,
          d.first_diag_date,
          d.first_end_date,
          c_std.concept_name        AS standard_concept_name,
          c_std.vocabulary_id       AS standard_vocabulary,
          c_type.concept_name       AS condition_type_concept_name,
          vis_evt.concept_name      AS visit_occurrence_concept_name
        FROM detail d
        JOIN `{dataset}.concept` c_std
          ON d.condition_concept_id = c_std.concept_id
        LEFT JOIN `{dataset}.concept` c_type
          ON d.condition_type_concept_id = c_type.concept_id
        LEFT JOIN `{dataset}.visit_occurrence` v
          ON d.visit_occurrence_id = v.visit_occurrence_id
        LEFT JOIN `{dataset}.concept` vis_evt
          ON v.visit_concept_id = vis_evt.concept_id
        WHERE d.rn = 1
      ),

      spec_metrics AS (
        SELECT
          person_id,
          MIN(condition_start_datetime) AS first_diag_date,
          MAX(condition_start_datetime) AS last_diag_date,
          COUNT(DISTINCT visit_occurrence_id) AS visits_for_concept
        FROM spec_occ
        GROUP BY person_id
      ),

      allv AS (
        SELECT
          person_id,
          COUNT(DISTINCT visit_occurrence_id) AS visits_all_concepts
        FROM all_occ
        GROUP BY person_id
      ),

      conc AS (
        SELECT
          person_id,
          COUNT(DISTINCT condition_concept_id) AS concept_count_ehr
        FROM all_occ
        GROUP BY person_id
      )

    SELECT
      fd.person_id,
      fd.condition_concept_id,
      fd.standard_concept_name,
      fd.standard_vocabulary,
      fd.first_diag_date,
      sm.last_diag_date,
      fd.condition_type_concept_name,
      fd.visit_occurrence_concept_name,
      sm.visits_for_concept,
      av.visits_all_concepts,
      cc.concept_count_ehr
    FROM first_detail fd
    JOIN spec_metrics sm  ON fd.person_id = sm.person_id
    LEFT JOIN allv av       ON fd.person_id = av.person_id
    LEFT JOIN conc cc       ON fd.person_id = cc.person_id
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    return df

In [ ]:
def demographics_table():
    """
    Fetch person demographics rows for a single concept_id,
    using your original SQL structure and injecting concept_id directly.
    """
    dataset = os.environ["WORKSPACE_CDR"]

    demographics_sql  = f"""
    SELECT
        person.person_id,
        
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
   
        p_race_concept.concept_name as race,
    
        p_ethnicity_concept.concept_name as ethnicity,
    
        p_sex_at_birth_concept.concept_name as sex_at_birth,
      
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `{dataset}.person` person 
    LEFT JOIN
        `{dataset}.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `{dataset}.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_array_data = 1 ) )"""


    demographics_df = pd.read_gbq(
        demographics_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook"
    )

    return demographics_df

In [ ]:
def get_vaccine_table(vaccine_ids):
    """
    Fetches drug exposure rows for the specified vaccine_concept_id(s),
    applying Cohort Builder UI filters (EHR + genomics, flat‐events,
    standard concepts), and returns a DataFrame with:
      - person_id
      - drug_concept_id
      - standard_concept_name
      - drug_exposure_start/end_datetime
      - verbatim_end_date
      - source_concept_name
    """
    dataset = os.environ["WORKSPACE_CDR"]
    # allow passing either a single int or a list/tuple of ints
    if not isinstance(vaccine_ids, (list, tuple)):
        vaccine_ids = [vaccine_ids]
    ids_sql = "(" + ",".join(str(i) for i in vaccine_ids) + ")"

    sql = f"""
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name AS drug_standard_concept_name,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_source_concept.concept_name AS source_concept_name 
    FROM (
        SELECT * 
        FROM `{dataset}.drug_exposure` d_exposure 
        WHERE
            drug_concept_id IN (
                SELECT DISTINCT ca.descendant_id 
                FROM `{dataset}.cb_criteria_ancestor` ca 
                JOIN (
                    SELECT DISTINCT c.concept_id       
                    FROM `{dataset}.cb_criteria` c       
                    JOIN (
                        SELECT CAST(cr.id AS STRING) AS id             
                        FROM `{dataset}.cb_criteria` cr             
                        WHERE
                            cr.concept_id IN {ids_sql}           
                            AND cr.full_text LIKE '%_rank1]%'       
                    ) a 
                      ON (
                        c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id
                      ) 
                    WHERE
                        c.is_standard   = 1 
                        AND c.is_selectable = 1
                ) b 
                  ON ca.ancestor_id = b.concept_id
            )
          AND d_exposure.person_id IN (
            SELECT DISTINCT p.person_id  
            FROM `{dataset}.cb_search_person` p  
            WHERE
                p.has_ehr_data = 1 
              AND (
                   p.has_whole_genome_variant    = 1 
                OR p.has_lr_whole_genome_variant = 1 
                OR p.has_array_data              = 1 
              )
          )
          AND d_exposure.person_id IN (
            SELECT criteria.person_id 
            FROM (
              SELECT DISTINCT person_id, entry_date, concept_id 
              FROM `{dataset}.cb_search_all_events` 
              WHERE
                concept_id IN (
                  SELECT DISTINCT c.concept_id 
                  FROM `{dataset}.cb_criteria` c 
                  JOIN (
                    SELECT CAST(cr.id AS STRING) AS id       
                    FROM `{dataset}.cb_criteria` cr       
                    WHERE
                        cr.concept_id IN (440029)       
                      AND cr.full_text LIKE '%_rank1]%'      
                  ) a 
                    ON (
                      c.path LIKE CONCAT('%.', a.id, '.%') 
                      OR c.path LIKE CONCAT('%.', a.id) 
                      OR c.path LIKE CONCAT(a.id, '.%') 
                      OR c.path = a.id
                    ) 
                  WHERE
                    c.is_standard   = 1 
                    AND c.is_selectable = 1
                )
                AND is_standard = 1
            ) criteria
          )
    ) d_exposure 
    LEFT JOIN `{dataset}.concept` d_standard_concept 
      ON d_exposure.drug_concept_id       = d_standard_concept.concept_id 
    LEFT JOIN `{dataset}.concept` d_source_concept 
      ON d_exposure.drug_source_concept_id = d_source_concept.concept_id
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    return df

In [ ]:
COND_NAME_MAP = dict(
    zip(seasonal_table['concept_id'], seasonal_table['standard_concept_name']))


def merge_cond_vax_table(s_vax_id_dict): 
    
    s_vax_cohort_dict = {}
    
    demo = demographics_table().drop_duplicates('person_id')
    
    for key, vax_list in s_vax_id_dict.items() :
    
        cond = get_condition_summary(key)
        vax = get_vaccine_table(vax_list)
        
        cohort_base = cond.merge(demo,  on='person_id', how='left')
        
        vax_cohort =cond.merge(vax , on='person_id', how='inner')
        
        pre_vax_df = vax_cohort.loc[vax_cohort['drug_exposure_start_datetime']  < vax_cohort['first_diag_date']].copy()
        
        pre_vax_df = (
            pre_vax_df
            .sort_values("drug_exposure_start_datetime")
            .drop_duplicates(subset="person_id", keep="first"))
        
        
        final_merge = cohort_base.merge(pre_vax_df, on = 'person_id', how = 'left' )
        
        #find dictionary keys, and add final tables to dict
        cond_name = COND_NAME_MAP.get(key, "<unknown>")
        s_vax_cohort_dict[(key, cond_name)] = final_merge
        
        
       

    return s_vax_cohort_dict


In [ ]:
s_vax_df = merge_cond_vax_table(cohort_vax_dict)
s_vax_df

In [ ]:
def merge_race_ethnicity_data(new_column):
    
    if new_column["ethnicity"] == "Hispanic or Latino":
        return new_column["ethnicity"]
    else:
        return new_column["race"]
    
for key, table in s_vax_df.items():
    
    table["updated_race"] = table.apply(merge_race_ethnicity_data, axis=1)
    
     

In [ ]:
def add_vax_count_by_ethnicity(new_column):
    
    if new_column["drug_exposure_start_datetime"] is pd.NaT:
        return 'N'
    else:
        return 'Y'
    
for key, table in s_vax_df.items():
    
    table["vaccinated"] = table.apply(add_vax_count_by_ethnicity, axis=1)

In [ ]:
s_vax_df

In [ ]:
import pickle
from pathlib import Path

# 1) Suppose `ns_vax_df` is your dict of DataFrames
#    Example: ns_vax_df = {'a': df1, 'b': df2, ...}

# 2) Choose a workspace folder for persistence
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/seasonal_vaccine_influenza_cohort_data_table.pkl')

# 3) Save the entire dict in one go
with open(out_file, 'wb') as f:
    pickle.dump(s_vax_df, f)

print(f"Saved {len(s_vax_df)} DataFrames to {out_file}")

In [ ]:
import pandas as pd

def get_vax_seasonal_binning(seasonal_vax_df):
    seasonal_vax_cohort_bin_dict = {}
    
    # define your epi‐season bounds
    start_week = 40   # week 40 of the season year
    end_week   = 20   # week 20 of the following year

    for key, table in seasonal_vax_df.items():
        # extract ISO week and year
        iso = table['first_diag_date_x'].dt.isocalendar()
        week = iso.week
        year = iso.year

        # assign each row to the season_year in which its season started:
        #  - if week >= start_week → belongs to season starting that calendar year
        #  - else                → belongs to season that started the previous calendar year
        table['season_year'] = year.where(week >= start_week, year - 1)

        # keep only rows in the seasonal window (week >= 40 OR week <= 20)
        in_window = (week >= start_week) | (week <= end_week)
        df_window = table.loc[in_window].copy()

        # count updated_race by season_year
        seasonal_vax_count = (
            df_window
              .groupby('season_year')[['updated_race','vaccinated']]
              .value_counts()
        )

        # turn into a DataFrame and attach concept info
        df_counts = seasonal_vax_count.rename('count').reset_index()
        df_counts['concept_id']   = key[0]
        df_counts['concept_name'] = key[1]

        seasonal_vax_cohort_bin_dict[key] = df_counts

    # concatenate all concepts’ results into one table
    seasonal_vax_table = pd.concat(
        seasonal_vax_cohort_bin_dict.values(),
        axis=0,
        ignore_index=True
    )

    return seasonal_vax_table

vax_seasonal_bin_counts = get_vax_seasonal_binning(s_vax_df)
vax_seasonal_bin_counts

count_filter = vax_seasonal_bin_counts.loc[vax_seasonal_bin_counts['count'] >= 100, :]
count_filter

def has_Y_and_N(group):
    vax_set = set(group['vaccinated'])
    return vax_set == {'Y', 'N'} and len(group) == 2

# Apply groupby and filter
filtered_df = count_filter.groupby(['season_year','updated_race', 'concept_id']).filter(has_Y_and_N)

df1 = pd.DataFrame(filtered_df)
df1


In [ ]:
df1.shape

In [ ]:
#getting person_id for each binn 

In [ ]:
import pandas as pd

# ─── 0) INPUT: s_vax_df ────────────────────────────────────────────────────────
# A dict mapping (concept_id, concept_name) → person-level DataFrame
# each DF must have at least:
#   ['person_id', 'first_diag_date_x' (datetime64), 'updated_race', 'vaccinated', …]

# ─── 1) FLATTEN AND ASSIGN SEASON • WINDOW ─────────────────────────────────────
records = []
for (cid, cname), df in s_vax_df.items():
    tmp = df.copy()
    iso  = tmp['first_diag_date_x'].dt.isocalendar()
    week = iso.week
    year = iso.year

    # season_year = year if week ≥ 40, else year - 1
    tmp['season_year'] = year.where(week >= 40, year - 1)

    # only keep weeks in [40..52] ∪ [1..20]
    in_window = (week >= 40) | (week <= 20)
    tmp = tmp.loc[in_window].copy()

    # tag concept
    tmp['concept_id']   = cid
    tmp['concept_name'] = cname

    records.append(tmp)

master = pd.concat(records, ignore_index=True)

# ─── 2) COUNT PER SLICE ─────────────────────────────────────────────────────────
seasonal_counts = (
    master
    .groupby(
        ['concept_id', 'concept_name', 'season_year', 'updated_race', 'vaccinated'],
        as_index=False
    )['person_id']
    .count()
    .rename(columns={'person_id': 'count'})
)

# ─── 3) FILTER TO count ≥ 100 ─────────────────────────────────────────────────
count_filter = seasonal_counts.loc[seasonal_counts['count'] >= 100].copy()

# ─── 4) KEEP ONLY GROUPS WITH BOTH Y & N ────────────────────────────────────────
def has_Y_and_N(g):
    s = set(g['vaccinated'])
    return s == {'Y', 'N'} and len(g) == 2

filtered = (
    count_filter
    .groupby(['concept_id', 'concept_name', 'season_year', 'updated_race'])
    .filter(has_Y_and_N)
    .reset_index(drop=True)
)

# ─── 5) BUILD DICT OF PERSON-LEVEL DFs ─────────────────────────────────────────
# Key = (concept_id, concept_name, season_year, updated_race, vaccinated)
seasonal_person_dfs = {}
for row in filtered.itertuples(index=False):
    key = (
        row.concept_id,
        row.concept_name,
        row.season_year,
        row.updated_race,
        row.vaccinated
    )
    sub = master[
        (master['concept_id']    == row.concept_id) &
        (master['concept_name']  == row.concept_name) &
        (master['season_year']   == row.season_year) &
        (master['updated_race']  == row.updated_race) &
        (master['vaccinated']    == row.vaccinated)
    ].copy()
    seasonal_person_dfs[key] = sub



In [ ]:
len(seasonal_person_dfs.keys())

In [ ]:
import pickle
from pathlib import Path

# 1) Suppose `ns_vax_df` is your dict of DataFrames
#    Example: ns_vax_df = {'a': df1, 'b': df2, ...}

# 2) Choose a workspace folder for persistence
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/seasonal_vaccine_influenza_seasonal_bin_cohort_data_table.pkl')

# 3) Save the entire dict in one go
with open(out_file, 'wb') as f:
    pickle.dump(seasonal_person_dfs, f)

print(f"Saved {len(seasonal_person_dfs)} DataFrames to {out_file}")

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# …later, in any notebook in the same workspace…
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/seasonal_vaccine_influenza_seasonal_bin_cohort_data_table.pkl')

# 4) Reload with one line:
with open(out_file, 'rb') as f:
    flu = pickle.load(f)

In [ ]:
for keys in flu.keys():
    print(keys)